# 강의 03 · 실습 4 — RAG 에이전트 서비스 · (3.5) 디버깅

## 1. 문제상황

- 시립도서관 홈페이지 팀은 도서관 FAQ 검색기를 안내 담당자 서비스로 만들어 붙이려 합니다.
- 손님 질문 중에는 「오늘 몇 시까지 해요」처럼 오늘이 무슨 요일인지 알아야 FAQ의 운영시간 항목으로 답할 수 있는 질문이 있습니다.
- FAQ에는 요일별 운영시간이 있지만 오늘이 무슨 요일인지는 없고, 모델도 오늘 날짜를 모릅니다.
- 담당자는 요일을 알려 주는 도구를 하나 더 두어, 모델이 요일 도구와 FAQ 검색 도구를 이어서 부르게 하고, 같은 주소로 서비스하기를 원합니다.

## 2. 문제와 목표

- **문제**: 오늘 요일이 있어야 답할 수 있는 질문을 FAQ 검색 하나로는 처리하지 못하고, 도서관 FAQ로 바뀐 검색기를 서비스로 노출하지 않았습니다. 아래 「6. 코드 — 스텝바이스텝」의 완성 코드는 이 목표를 잘못 구현해 문법 오류 없이 실행되지만 결과가 요구사항과 어긋나며, 결함 세 개를 찾아 고쳐 「7. 실행 결과 확인」을 통과시키는 것이 이 단의 과제입니다.
- **목표**: 도서관 FAQ 12행으로 검색기를 세우고, 요일 도구 `today_weekday`와 검색 도구 `faq_search` 두 개를 쥔 안내 담당자를 만들어, 한 질문에 도구 두 개를 이어 부르는 것을 확인하고, `POST /ask`로 노출합니다.
    - 도서관 FAQ: `library_faq.csv` 12행, 저장소 디렉터리 `chroma_library`.
    - 도구 두 개: today_weekday(인자 없음, 평일·토요일·일요일 중 하나), faq_search(임계값 1.5).
    - 안내 담당자: 두 도구를 묶은 손 루프 run_rag_agent — 도구 이름으로 도구를 찾는 사전 `TOOLS`로 실행합니다.
- **목표 달성 여부의 판정 기준**: 「오늘 몇 시까지 해요?」에서 요일 도구 호출 뒤 검색 호출이 이어져 두 바퀴를 돌고 최종 답에 오늘 요일의 운영시간이 들어 있으며, 「책 몇 권까지 빌려요?」는 검색 한 번으로 답하고, 문서 밖 질문은 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」로 끝나는 것을 노트북과 서비스 호출 양쪽에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex04_s3_diagram.svg)

## 4. 단계별 요구사항

1. **문서를 적재합니다.**
    - `library_faq.csv`(12행)를 `csv.DictReader`로 읽고(`utf-8-sig`), `Question`이 빈 행은 버립니다.
    - 행마다 `[카테고리] Q: 질문\nA: 답변` 형식의 본문과 `{"row": 행 번호, "category": 카테고리}` 메타데이터를 가진 `Document`를 만들어 리스트 `docs`에 모으고, 적재 문서 수를 출력합니다.
2. **임베딩을 준비합니다.**
    - `OpenAIEmbeddings(model="text-embedding-3-small")`로 임베딩 부품 `emb`를 만듭니다.
3. **저장소를 구축하고 영속합니다.**
    - `Chroma.from_documents(docs, emb, persist_directory="chroma_library", ids=[...])`로 저장소 `db`를 만들고, 문서 id는 `row-<행 번호>`로 줍니다.
4. **점수와 함께 검색합니다.**
    - `retrieve(query, k=2)`로 `db.similarity_search_with_score`의 청크·점수 목록을 돌려주고, 「몇 권까지 빌려요」로 시험해 점수를 출력합니다.
5. **검색 도구와 요일 도구를 선언합니다.**
    - `THRESHOLD = 1.5`. `@tool(parse_docstring=True)`를 붙인 `faq_search(query)`는 `retrieve`로 얻은 청크 중 점수가 임계값 이하인 것만 `[score …] 본문` 형식으로 이어 돌려주고, 하나도 없으면 「검색 결과 없음 (최고 유사도 점수 X가 임계값 1.5를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라.」를 돌려줍니다.
    - 독스트링에 도구 설명 「도서관 FAQ 문서에서 질문과 관련된 청크를 검색한다. 이용안내, 대출, 시설 관련 질문에 쓴다.」와 `query` 인자 설명을 적습니다.
    - 여기에 더해, 인자 없이 오늘 요일을 「평일」「토요일」「일요일」 중 하나로 돌려주는 `today_weekday()`를 `@tool`로 선언합니다(월~금은 「평일」). 독스트링에 「오늘이 무슨 요일인지 알아야 하는 질문에 먼저 쓴다」를 적습니다.
6. **입출력 모양과 고정 문장을 선언합니다.**
    - `AskIn(question: str)`, `AskOut(answer: str)`, 고정 안내 문장 `NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."`, 시스템 프롬프트 `SYSTEM`을 둡니다.
7. **처리 함수를 만듭니다.**
    - `run_rag_agent(question, max_turn=5)`는 시스템 프롬프트와 질문으로 대화 기록을 만들고, `llm.bind_tools([faq_search, today_weekday])` 손 루프를 돕니다.
    - 도구 이름으로 도구를 찾는 사전 `TOOLS`로 도구를 실행하고 결과를 `ToolMessage`로 되먹입니다.
    - 도구 호출이 없으면 답을 돌려주되, 검색을 했는데 근거가 한 번도 없었으면(`searched and not found`) `NO_EVIDENCE`를 돌려줍니다.
    - `found`는 `faq_search`의 결과에만 적용합니다.
    - 도구 호출과 결과를 출력하고, 상한에 닿으면 「반복 한도 초과」를 돌려줍니다.
    - 「오늘 몇 시까지 해요?」「책 몇 권까지 빌려요?」「파이썬 리스트 정렬은 어떻게 하나요?」로 시험합니다.
8. **앱과 엔드포인트를 등록합니다.**
    - `%%writefile app_s3-5.py`로 서비스 파일을 만듭니다.
    - 파일은 `.env`를 읽고 `chroma_library`를 다시 열어 4~7번의 도구 두 개·처리 함수를 그대로 담고, `app = FastAPI()`, `GET /healthz`, `POST /ask`(`AskIn`을 받아 `AskOut(answer=run_rag_agent(...))`를 돌려줌)를 등록합니다.
9. **기동하고 호출을 확인합니다.**
    - `fastapi dev app_s3-5.py --port 8031 --no-reload`를 `subprocess`로 띄우고 `/healthz`가 200을 줄 때까지 기다린 뒤, `samples_library.json`의 질문 세 개를 `httpx.post("/ask")`로 보내 상태 코드와 응답 JSON을 출력하고, 서버를 종료합니다.
    - 처리 함수는 도구를 부를 때 「[도구 호출] 도구 이름 인자 → 결과 앞부분」 줄을, 서버 확인은 「[healthz] 200 ok」 줄을 출력합니다.
    - 서비스 주소는 `http://127.0.0.1:8031`입니다.

## 5. 코드 골격 — RAG 인덱싱·검색 5단 + FastAPI 서비스 4단

두 골격을 잇습니다. ⑤에 검색 도구와 요일 도구가, 서비스 ②에 도구 사전이 있습니다.

**RAG 인덱싱·검색 5단** (검색 도구 안에 접힙니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 문서 적재 | 원본 파일을 읽어 검색 단위 문서로 만듭니다 | `Document(page_content=..., metadata=...)` | 1 |
| ② 임베딩 준비 | 문장을 숫자 벡터로 바꿀 모델을 지정합니다 | `OpenAIEmbeddings(model="text-embedding-3-small")` | 2 |
| ③ 저장소 구축·영속 | 문서와 임베딩을 넣어 저장소를 만들고 디렉터리에 남깁니다 | `Chroma.from_documents(docs, emb, persist_directory=...)` | 3 |
| ④ 점수 동반 검색 | 질문을 넣어 가까운 문서와 그 거리 점수를 함께 받습니다 | `db.similarity_search_with_score(q, k=2)` | 4 |
| ⑤ 임계값 컷 | 점수가 기준을 넘으면 문서 근거를 쓰지 않고 다른 경로로 보냅니다 | `@tool faq_search` 안의 `if s <= THRESHOLD` | 5 |

**FastAPI 서비스 4단** (서비스 표준 템플릿 `service_template`의 구성과 같습니다)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| 서비스 ① 입출력 모양 선언 | 서비스가 받을 값과 돌려줄 값의 모양을 클래스로 선언합니다 | `class AskIn(BaseModel)`, `class AskOut(BaseModel)` | 6 |
| 서비스 ② 처리 함수 구현 | 값을 받아 결과를 돌려주는 함수를 웹과 무관하게 먼저 만듭니다 | `def run_rag_agent(question) -> str`, `TOOLS = {...}` | 7 |
| 서비스 ③ 앱·엔드포인트 등록 | 앱을 만들고, 주소와 함수를 데코레이터로 잇습니다 | `app = FastAPI()`, `@app.post("/ask")` | 8 |
| 서비스 ④ 기동·호출 확인 | 개발 서버를 띄우고 요청을 보내 응답을 확인합니다 | `fastapi dev app_s3-5.py --port ...`, `httpx.post("/ask")` | 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다. 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.
- 서비스 파일 `app_s3-5.py`도 같은 `.env`를 `find_dotenv(usecwd=True)`로 읽습니다.

In [ ]:
import csv
import json
import os
import subprocess
import sys
import time

import httpx
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 문서 적재 (요구사항 1)

- FAQ 한 행을 청크(chunk) 하나로 삼고, 질문과 답을 한 본문에 넣습니다.

In [ ]:
CSV_PATH = "library_faq.csv"

docs = []
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    for i, row in enumerate(csv.DictReader(f), start=1):
        if not (row.get("Question") or "").strip():
            continue
        text = f"[{row['Category']}] Q: {row['Question']}\nA: {row['Answer']}"
        docs.append(Document(page_content=text, metadata={"row": i, "category": row["Category"]}))
print("적재 문서 수:", len(docs))

### 단계 ② — 임베딩 준비 (요구사항 2)

- 문장을 숫자 벡터로 바꿀 임베딩 모델을 지정합니다.

In [ ]:
emb = OpenAIEmbeddings(model="text-embedding-3-small")
print("임베딩 준비를 마쳤습니다.")

### 단계 ③ — 저장소 구축·영속 (요구사항 3)

- 저장소를 디렉터리에 남깁니다. 서비스 파일 `app_s3-5.py`는 이 디렉터리를 다시 열어 씁니다.

In [ ]:
db = Chroma.from_documents(
    docs, emb,
    persist_directory="chroma_library",
    ids=[f"row-{d.metadata['row']}" for d in docs],
)
print("저장된 항목 수:", len(db.get()["ids"]))

### 단계 ④ — 점수 동반 검색 (요구사항 4)

- 점수 동반 검색을 함수 `retrieve`로 감쌉니다. 도구는 이 함수를 부릅니다.

In [ ]:
def retrieve(query: str, k: int = 2):
    """청크와 거리 점수의 목록을 돌려준다."""
    return db.similarity_search_with_score(query, k=k)


for d, s in retrieve("몇 권까지 빌려요"):
    print(f"  score={s:.4f} | {d.page_content[:50].replace(chr(10), ' / ')}")

### 단계 ⑤ — 임계값 컷을 품은 검색 도구 (요구사항 5)

- 임계값 컷이 도구 안에 들어갑니다. 도구는 임계값 안의 청크만 돌려주고, 없으면 「검색 결과 없음」 문장을 돌려줍니다. 이 문장이 모델에게 되먹여집니다.
- 독스트링이 도구 설명입니다. 모델은 이 설명을 보고 사실 질문에 도구를 부릅니다.
- 요일 도구 `today_weekday`는 인자가 없고 오늘 요일 문자열을 돌려줍니다. 독스트링의 「먼저 쓴다」가 모델에게 호출 순서를 알립니다.

In [ ]:
THRESHOLD = 1.5


@tool(parse_docstring=True)
def faq_search(query: str) -> str:
    """도서관 FAQ 문서에서 질문과 관련된 청크를 검색한다. 이용안내, 대출, 시설 관련 질문에 쓴다.

    Args:
        query: 검색할 질의문. 결과가 없으면 표현을 바꿔 재검색할 수 있다.
    """
    hits = retrieve(query)
    good = [(d, s) for d, s in hits if s >= THRESHOLD]
    if not good:
        return f"검색 결과 없음 (최고 유사도 점수 {hits[0][1]:.2f}가 임계값 {THRESHOLD}를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라."
    return "\n---\n".join(f"[score {s:.2f}] {d.page_content}" for d, s in good)


@tool
def today_weekday() -> str:
    """오늘이 평일, 토요일, 일요일 중 무엇인지 돌려준다. 오늘이 무슨 요일인지 알아야 하는 질문에 먼저 쓴다."""
    from datetime import date
    return ["평일", "평일", "평일", "평일", "평일", "토요일", "일요일"][date.today().weekday()]


TOOLS = {t.name: t for t in [faq_search, today_weekday]}
print("[도구]", list(TOOLS), "| 오늘:", today_weekday.invoke({}))

### 서비스 단계 ① — 입출력 모양 선언 (요구사항 6)

- 요청 본문과 응답 본문의 모양을 pydantic 클래스로 못 박습니다. 고정 안내 문장과 시스템 프롬프트도 여기서 선언합니다.
- 시스템 프롬프트는 매 호출 그대로 들어가며, 검색할지 말지와 근거 없을 때의 태도를 정합니다.

In [ ]:
class AskIn(BaseModel):
    question: str


class AskOut(BaseModel):
    answer: str


NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시립도서관 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "오늘 요일이 필요하면 today_weekday를 먼저 부른다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")
print("요청 칸:", list(AskIn.model_fields), "| 응답 칸:", list(AskOut.model_fields))

### 서비스 단계 ② — 처리 함수 구현 (요구사항 7)

- 웹과 무관한 파이썬 함수로 먼저 만들고 노트북에서 시험합니다. 모델이 도구 호출을 돌려주면 도구를 실행해 결과를 되먹이고, 도구 호출이 없으면 답을 돌려주는 손 루프입니다. 도구가 둘이므로 도구 이름으로 도구를 찾는 사전이 필요합니다.
- 검색을 했는데 근거가 한 번도 없었으면 모델의 답 대신 고정 안내 문장을 돌려줍니다. 「모른다고 답하라」를 모델 재량에만 맡기지 않습니다.

In [ ]:
llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
llm_tools = llm.bind_tools([faq_search, today_weekday])


def run_rag_agent(question: str, max_turn: int = 1) -> str:
    """도구 두 개를 쥔 손 루프. 근거 없음 하드 가드와 반복 상한이 있다."""
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    searched, found = False, False
    for _ in range(max_turn):
        res = llm_tools.invoke(messages)
        if not res.tool_calls:
            if searched and not found:
                return NO_EVIDENCE
            return res.content
        messages.append(res)
        for call in res.tool_calls:
            result = TOOLS[call["name"]].invoke(call["args"])
            if call["name"] == "faq_search":
                searched = True
                if not result.startswith("검색 결과 없음"):
                    found = True
            print(f"  [도구 호출] {call['name']} {call['args']} → {str(result)[:40].replace(chr(10), ' ')}…")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "반복 한도 초과"


for q in ["오늘 몇 시까지 해요?", "책 몇 권까지 빌려요?", "파이썬 리스트 정렬은 어떻게 하나요?"]:
    print(f"[질문] {q}")
    print(f"  [답] {run_rag_agent(q)[:120]}")

### 서비스 단계 ③ — 앱·엔드포인트 등록 (요구사항 8)

- 서비스 파일은 노트북과 따로 도는 프로그램이므로 저장소를 다시 열고 도구·처리 함수를 파일 안에 그대로 둡니다.
- `@app.post("/ask")`가 주소와 함수를 잇습니다. 함수 안에서 처리 함수를 그대로 부르고, 반환한 값이 JSON 응답 본문이 됩니다.

In [ ]:
%%writefile app_s3-5.py
# -*- coding: utf-8 -*-
"""시립도서관 RAG 에이전트 서비스 — 서비스 표준 템플릿 구성 (실행: fastapi dev app_s3-5.py --port 8031)."""

import os

from dotenv import find_dotenv, load_dotenv
from fastapi import FastAPI
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from pydantic import BaseModel

load_dotenv(find_dotenv(usecwd=True))

THRESHOLD = 1.5
NO_EVIDENCE = "문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요."
SYSTEM = ("너는 시립도서관 안내 담당자다. 인사말처럼 검색이 필요 없는 말에는 바로 답한다. "
          "오늘 요일이 필요하면 today_weekday를 먼저 부른다. "
          "그 밖의 모든 질문은 반드시 faq_search 도구로 근거를 먼저 찾고, 도구 결과에 있는 내용으로만 답한다. "
          "도구 결과가 '검색 결과 없음'이면 네가 아는 지식으로 답하지 말고 "
          f"'{NO_EVIDENCE}'라고만 답한다.")

emb = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(persist_directory=os.path.join(os.path.dirname(os.path.abspath(__file__)), "chroma_library"), embedding_function=emb)


def retrieve(query: str, k: int = 2):
    return db.similarity_search_with_score(query, k=k)


@tool(parse_docstring=True)
def faq_search(query: str) -> str:
    """도서관 FAQ 문서에서 질문과 관련된 청크를 검색한다. 이용안내, 대출, 시설 관련 질문에 쓴다.

    Args:
        query: 검색할 질의문. 결과가 없으면 표현을 바꿔 재검색할 수 있다.
    """
    hits = retrieve(query)
    good = [(d, s) for d, s in hits if s <= THRESHOLD]
    if not good:
        return f"검색 결과 없음 (최고 유사도 점수 {hits[0][1]:.2f}가 임계값 {THRESHOLD}를 넘음). 질의를 바꿔 다시 검색하거나, 모른다고 답하라."
    return "\n---\n".join(f"[score {s:.2f}] {d.page_content}" for d, s in good)


@tool
def today_weekday() -> str:
    """오늘이 평일, 토요일, 일요일 중 무엇인지 돌려준다. 오늘이 무슨 요일인지 알아야 하는 질문에 먼저 쓴다."""
    from datetime import date
    return ["평일", "평일", "평일", "평일", "평일", "토요일", "일요일"][date.today().weekday()]


TOOLS = {t.name: t for t in [faq_search, today_weekday]}
llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
llm_tools = llm.bind_tools([faq_search, today_weekday])


# ── ① 입출력 모양 선언
class AskIn(BaseModel):
    question: str


class AskOut(BaseModel):
    answer: str


# ── ② 처리 함수 구현
def run_rag_agent(question: str, max_turn: int = 5) -> str:
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    searched, found = False, False
    for _ in range(max_turn):
        res = llm_tools.invoke(messages)
        if not res.tool_calls:
            if searched and not found:
                return NO_EVIDENCE
            return res.content
        messages.append(res)
        for call in res.tool_calls:
            result = TOOLS[call["name"]].invoke(call["args"])
            if call["name"] == "faq_search":
                searched = True
                if not result.startswith("검색 결과 없음"):
                    found = True
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "반복 한도 초과"


# ── ③ 앱 생성·엔드포인트 등록
app = FastAPI(title="시립도서관 RAG 에이전트 서비스")


@app.get("/healthz")
def healthz() -> dict:
    return {"status": "ok"}


@app.get("/ask", response_model=AskOut)
def ask_endpoint(req: AskIn) -> AskOut:
    return AskOut(answer=run_rag_agent(req.question))


# ── ④ 기동·호출 확인: 터미널에서 `fastapi dev app_s3-5.py --port 8031` 뒤 `python run_samples.py 8031`

### 서비스 단계 ④ — 기동·호출 확인 (요구사항 9)

- 노트북에서 개발 서버를 자식 프로세스로 띄우고, 살아 있는지 확인한 뒤 요청을 보내고, 끝나면 서버를 내립니다.
- 터미널에서는 `fastapi dev app_s3-5.py --port 8031`로 띄우고 다른 터미널에서 `python run_samples.py 8031`로 같은 확인을 합니다. PowerShell의 `curl`은 별칭이므로 `curl.exe`나 `run_samples.py`를 씁니다.
- 포트 8031이 이미 쓰이고 있으면 노트북과 명령의 포트 숫자를 함께 바꿉니다.

In [ ]:
BASE = "http://127.0.0.1:8031"
import tempfile
LOG_PATH = os.path.join(tempfile.gettempdir(), "lec03_ex04_server_stderr.log")   # 기동 실패 원인을 보는 서버 오류 출력 파일
log = open(LOG_PATH, "w", encoding="utf-8")
server = subprocess.Popen([sys.executable, "-m", "fastapi", "dev", "app_s3-5.py", "--port", "8031", "--no-reload"],
                          stdout=subprocess.DEVNULL, stderr=log,
                          env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1"))   # cp949 콘솔에서도 서버가 죽지 않게
started = False
try:
    for _ in range(60):                  # 2초 × 60회 = 최대 120초 대기
        try:
            if httpx.get(f"{BASE}/healthz", timeout=2).status_code == 200:
                started = True
                break
        except httpx.HTTPError:
            pass
        time.sleep(2)
    if not started:
        log.flush()
        tail = open(LOG_PATH, encoding="utf-8").read()[-1500:]
        raise RuntimeError(f"서버가 120초 안에 뜨지 않았습니다. 서버 오류 출력({LOG_PATH}): " + tail)
    print("[healthz] 200 ok")

    samples = json.load(open("samples_library.json", encoding="utf-8"))
    for i, payload in enumerate(samples, 1):
        r = httpx.post(f"{BASE}/ask", json=payload, timeout=120)
        print(f"[{i}] 요청: {payload['question']} | 상태: {r.status_code}")
        print(f"[{i}] 응답: {r.json()}")
finally:
    server.terminate()
    server.wait(timeout=10)
    log.close()
    if started:
        try:
            os.remove(LOG_PATH)
        except OSError:
            pass                     # 자식 프로세스가 아직 파일을 쥐고 있으면 임시 폴더에 그대로 둔다
    print("서버를 내렸습니다.")

## 7. 실행 결과 확인

결함을 고친 뒤 다시 실행해 다음 세 가지를 확인합니다.

1. 「책 몇 권까지 빌려요?」가 「5권까지 14일」이 들어간 답으로 끝나고, 파이썬 질문이 고정 안내 문장으로 끝납니다.
2. 「오늘 몇 시까지 해요?」가 「반복 한도 초과」가 아니라 오늘 요일의 운영시간으로 끝납니다.
3. 서비스 단계 ④에서 세 요청의 상태가 모두 200입니다.

고치기 전에는 세 질문이 모두 「반복 한도 초과」로 끝나고, 서비스 호출의 상태가 405입니다. 반복 상한을 고치면 문서 안 질문이 고정 안내 문장으로 끝나는 증상이 드러납니다. 증상 하나를 고치면 다음 증상이 나옵니다.